# Mean Group & Pooled Mean Group Estimators

This notebook introduces the **Mean Group (MG)** and **Pooled Mean Group (PMG)** estimators for panel data with heterogeneous slopes.

**Key references:**
- Pesaran & Smith (1995) — Mean Group estimator
- Pesaran, Shin & Smith (1999) — Pooled Mean Group estimator
- Swamy (1970) — Slope homogeneity test

**Topics covered:**
1. Why slope heterogeneity matters
2. Simulation: FE bias under heterogeneous slopes
3. Mean Group estimation with Grunfeld data
4. Entity-specific coefficients and visualization
5. Swamy test for slope homogeneity
6. Pooled Mean Group (PMG) estimation
7. Hausman test: MG vs PMG
8. Comparison: FE vs RE vs MG vs PMG
9. Exercises

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import panelbox as pb

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

---
## 1. Why Slope Heterogeneity Matters

Standard panel estimators (Fixed Effects, Random Effects) assume that the slope coefficients are **the same across all entities**:

$$y_{it} = \alpha_i + \beta' x_{it} + \varepsilon_{it}$$

But in many applications, the relationship between $y$ and $x$ varies across entities. For example, the effect of market value on investment may differ across firms due to different technologies, management strategies, or industry sectors.

When slopes are truly heterogeneous:
$$y_{it} = \alpha_i + \beta_i' x_{it} + \varepsilon_{it}$$

the FE estimator converges to a **weighted average** of the $\beta_i$, but the weights depend on the within-entity variance of $x_{it}$ — not equal weights. This can produce **biased and inconsistent** estimates of the average effect.

---
## 2. Simulation: FE Bias Under Heterogeneous Slopes

Let's simulate a panel where each entity has its own slope and see how FE performs vs. the true average.

In [ ]:
# Simulate heterogeneous panel
N, T = 50, 30
true_beta_mean = 2.0
true_beta_std = 1.5  # substantial heterogeneity

rows = []
betas_true = {}
for i in range(1, N + 1):
    beta_i = np.random.normal(true_beta_mean, true_beta_std)
    alpha_i = np.random.normal(0, 5)
    betas_true[i] = beta_i
    for t in range(1, T + 1):
        # x correlated with entity effect (endogeneity that FE handles)
        x = alpha_i * 0.3 + np.random.normal(0, 2)
        y = alpha_i + beta_i * x + np.random.normal(0, 1)
        rows.append({'entity': i, 'time': t, 'y': y, 'x': x})

sim_data = pd.DataFrame(rows)

print(f"True average beta: {true_beta_mean:.2f}")
print(f"True beta std: {true_beta_std:.2f}")
print(f"Sample mean of beta_i: {np.mean(list(betas_true.values())):.4f}")

In [ ]:
# FE estimation
fe_sim = pb.FixedEffects('y ~ x', sim_data, 'entity', 'time')
fe_sim_res = fe_sim.fit()

# MG estimation
mg_sim = pb.MeanGroupEstimator('y ~ x', sim_data, 'entity', 'time', min_obs_per_entity=5)
mg_sim_res = mg_sim.fit()

true_avg = np.mean(list(betas_true.values()))

print(f"True average slope:  {true_avg:.4f}")
print(f"FE estimate:         {fe_sim_res.params['x']:.4f}")
print(f"MG estimate:         {mg_sim_res.params['x']:.4f}")
print(f"\nFE bias:             {fe_sim_res.params['x'] - true_avg:.4f}")
print(f"MG bias:             {mg_sim_res.params['x'] - true_avg:.4f}")

In [ ]:
# Distribution of entity-specific slopes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# True betas
true_vals = list(betas_true.values())
axes[0].hist(true_vals, bins=15, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(true_avg, color='red', linestyle='--', linewidth=2, label=f'True mean = {true_avg:.2f}')
axes[0].axvline(fe_sim_res.params['x'], color='orange', linestyle='-.', linewidth=2, label=f'FE = {fe_sim_res.params["x"]:.2f}')
axes[0].axvline(mg_sim_res.params['x'], color='green', linestyle=':', linewidth=2, label=f'MG = {mg_sim_res.params["x"]:.2f}')
axes[0].set_title('True Entity-Specific Slopes')
axes[0].set_xlabel('Beta')
axes[0].legend()

# MG estimated betas
mg_coefs = mg_sim_res.coefficient_table()['x'].values
axes[1].hist(mg_coefs, bins=15, alpha=0.7, color='seagreen', edgecolor='black')
axes[1].axvline(mg_sim_res.params['x'], color='red', linestyle='--', linewidth=2, label=f'MG mean = {mg_sim_res.params["x"]:.2f}')
axes[1].set_title('MG Estimated Entity-Specific Slopes')
axes[1].set_xlabel('Beta')
axes[1].legend()

plt.tight_layout()
plt.show()

The MG estimator recovers the true average slope more accurately than FE, because it gives **equal weight** to each entity's OLS estimate rather than weighting by within-entity variance.

---
## 3. Mean Group Estimation with Grunfeld Data

The Grunfeld (1958) dataset contains investment data for 10 large US firms over 20 years (1935-1954). Let's apply the MG estimator to this classic panel.

In [ ]:
data = pb.load_grunfeld()
print(f"Panel: {data['firm'].nunique()} firms x {data['year'].nunique()} years = {len(data)} observations")
data.head(10)

In [ ]:
# Fit Mean Group estimator
mg = pb.MeanGroupEstimator(
    formula='invest ~ value + capital',
    data=data,
    entity_col='firm',
    time_col='year',
    min_obs_per_entity=5
)
mg_results = mg.fit()
print(mg_results.summary())

**Key observations:**
- The MG coefficients are the **simple average** of 10 firm-specific OLS regressions
- Standard errors reflect **cross-firm dispersion** in the estimated slopes
- The Swamy test (shown at the bottom) tests whether slopes are truly heterogeneous

---
## 4. Entity-Specific Coefficients

A key advantage of MG is access to the full distribution of entity-specific estimates.

In [ ]:
# Coefficient table: all entities
coef_table = mg_results.coefficient_table()
coef_table

In [ ]:
# Summary for a single entity
print(mg_results.entity_summary(1))

In [ ]:
# Scatter plot: entity-specific coefficients with MG average
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, var in enumerate(['value', 'capital']):
    coefs = coef_table[var]
    mg_val = mg_results.params[var]
    
    axes[idx].barh(range(len(coefs)), coefs.values, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].axvline(mg_val, color='red', linestyle='--', linewidth=2, label=f'MG mean = {mg_val:.4f}')
    axes[idx].axvline(0, color='black', linewidth=0.5)
    axes[idx].set_yticks(range(len(coefs)))
    axes[idx].set_yticklabels([f'Firm {e}' for e in coefs.index])
    axes[idx].set_xlabel('Coefficient')
    axes[idx].set_title(f'Entity-Specific Coefficients: {var}')
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Built-in boxplot method
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
mg_results.plot_coefficient_distribution('value', ax=axes[0])
mg_results.plot_coefficient_distribution('capital', ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Entity-specific R-squared values
rsq_df = pd.DataFrame({
    'R-squared': mg_results.entity_rsquared,
}).sort_values('R-squared', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    [f'Firm {e}' for e in rsq_df.index],
    rsq_df['R-squared'],
    color='steelblue', alpha=0.7, edgecolor='black'
)
ax.set_xlabel('R-squared')
ax.set_title('Entity-Specific R-squared (OLS per firm)')
ax.set_xlim(0, 1)
for bar, val in zip(bars, rsq_df['R-squared']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')
plt.tight_layout()
plt.show()

---
## 5. Swamy Test for Slope Homogeneity

The **Swamy (1970) test** checks whether the entity-specific slopes are statistically different from each other:

- $H_0$: $\beta_1 = \beta_2 = \ldots = \beta_N$ (slopes are homogeneous)
- $H_1$: At least one $\beta_i$ differs

The test statistic is:
$$S = \sum_{i=1}^{N} (\hat{\beta}_i - \hat{\beta}_{FE})' \hat{V}_i^{-1} (\hat{\beta}_i - \hat{\beta}_{FE}) \sim \chi^2((N-1)K)$$

In [ ]:
swamy = mg_results.swamy_test_result

print("Swamy (1970) Slope Homogeneity Test")
print("=" * 40)
print(f"Statistic: {swamy['statistic']:.4f}")
print(f"p-value:   {swamy['pvalue']:.6f}")
print(f"df:        {swamy['df']}")
print(f"\nConclusion: {'Reject H0 -- slopes are heterogeneous' if swamy['reject'] else 'Fail to reject H0 -- slopes may be homogeneous'}")

The Swamy test strongly rejects slope homogeneity, confirming that the standard FE/RE assumption of common slopes is violated in the Grunfeld data. The MG estimator is more appropriate here.

---
## 6. Pooled Mean Group (PMG)

The **PMG estimator** (Pesaran, Shin & Smith, 1999) is a middle ground between FE and MG. It estimates an error-correction model (ECM) where:

- **Long-run coefficients** ($\theta$) are constrained to be **homogeneous** across entities
- **Short-run dynamics** ($\delta_i$) and **error-correction speeds** ($\phi_i$) are **entity-specific**

The ECM form:
$$\Delta y_{it} = \phi_i (y_{i,t-1} - \theta' x_{i,t-1}) + \delta_i' \Delta x_{it} + \mu_i + \varepsilon_{it}$$

where $\phi_i < 0$ ensures convergence to the long-run equilibrium.

In [ ]:
# Fit PMG
pmg = pb.PooledMeanGroupEstimator(
    formula='invest ~ value + capital',
    data=data,
    entity_col='firm',
    time_col='year',
    lags=1
)
pmg_results = pmg.fit()
print(pmg_results.summary())

In [ ]:
# Long-run coefficients (homogeneous)
print("Long-run coefficients (homogeneous across firms):")
print(pmg_results.long_run_params)
print()

# Error-correction speeds (entity-specific)
phi_df = pd.DataFrame({
    'phi_i': pmg_results.entity_phi,
}).sort_values('phi_i')

print("Error-correction speeds (phi_i):")
print(phi_df)
print(f"\nAll phi_i negative? {all(v < 0 for v in pmg_results.entity_phi.values())}")
print("(Negative phi confirms error-correction mechanism is active)")

In [ ]:
# Visualize error-correction speeds
fig, ax = plt.subplots(figsize=(10, 5))
entities = list(phi_df.index)
phis = phi_df['phi_i'].values
colors = ['firebrick' if p > -0.1 else 'steelblue' for p in phis]

ax.barh([f'Firm {e}' for e in entities], phis, color=colors, alpha=0.7, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.5)
ax.axvline(np.mean(phis), color='red', linestyle='--', label=f'Mean = {np.mean(phis):.3f}')
ax.set_xlabel('Error-correction speed (phi_i)')
ax.set_title('PMG: Entity-Specific Error-Correction Speeds')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Hausman Test: MG vs PMG

The **Hausman test** compares MG (consistent under heterogeneity) with PMG (efficient under long-run homogeneity):

- $H_0$: Long-run homogeneity holds $\Rightarrow$ PMG is efficient
- $H_1$: Long-run homogeneity fails $\Rightarrow$ Only MG is consistent

$$H = (\hat{\theta}_{MG} - \hat{\theta}_{PMG})' (V_{MG} - V_{PMG})^{-1} (\hat{\theta}_{MG} - \hat{\theta}_{PMG}) \sim \chi^2(K)$$

In [ ]:
hausman = pb.hausman_mg_pmg(mg_results, pmg_results)

print("Hausman Test: MG vs PMG")
print("=" * 40)
print(f"Statistic: {hausman['statistic']:.4f}")
print(f"p-value:   {hausman['pvalue']:.4f}")
print(f"df:        {hausman['df']}")
print()
if hausman['reject']:
    print("Reject H0: Long-run homogeneity is invalid.")
    print("Prefer MG (consistent estimator).")
else:
    print("Fail to reject H0: Long-run homogeneity is plausible.")
    print("PMG may be preferred (more efficient).")

---
## 8. Comparison: FE vs RE vs MG vs PMG

Let's compare all four estimators side by side.

In [ ]:
# Fit FE and RE for comparison
fe = pb.FixedEffects('invest ~ value + capital', data, 'firm', 'year')
fe_results = fe.fit()

re = pb.RandomEffects('invest ~ value + capital', data, 'firm', 'year')
re_results = re.fit()

In [ ]:
# Comparison table
comparison = pd.DataFrame({
    'FE': fe_results.params,
    'RE': re_results.params,
    'MG': mg_results.params,
}).round(4)

# Add PMG long-run params
pmg_lr = pmg_results.long_run_params
comparison['PMG (LR)'] = np.nan
for var in pmg_lr.index:
    if var in comparison.index:
        comparison.loc[var, 'PMG (LR)'] = round(pmg_lr[var], 4)

print("Coefficient Comparison")
print("=" * 60)
print(comparison.to_string())
print()
print("Notes:")
print("- FE and RE assume homogeneous slopes")
print("- MG allows fully heterogeneous slopes")
print("- PMG constrains long-run coefficients to be homogeneous")

In [ ]:
# Visual comparison of slope estimates
variables = ['value', 'capital']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, var in enumerate(variables):
    ax = axes[idx]
    
    # Entity-specific MG estimates
    mg_coefs = coef_table[var]
    ax.scatter(range(len(mg_coefs)), mg_coefs.values, color='steelblue', s=80, 
               zorder=3, label='MG (entity-specific)', alpha=0.8)
    
    # Horizontal lines for pooled estimators
    ax.axhline(mg_results.params[var], color='steelblue', linestyle='--', linewidth=2, label=f'MG mean = {mg_results.params[var]:.4f}')
    
    if var in fe_results.params.index:
        ax.axhline(fe_results.params[var], color='orange', linestyle='-.', linewidth=2, label=f'FE = {fe_results.params[var]:.4f}')
    
    if var in re_results.params.index:
        ax.axhline(re_results.params[var], color='green', linestyle=':', linewidth=2, label=f'RE = {re_results.params[var]:.4f}')
    
    if var in pmg_lr.index:
        ax.axhline(pmg_lr[var], color='purple', linestyle='-', linewidth=1.5, label=f'PMG LR = {pmg_lr[var]:.4f}')
    
    ax.set_xticks(range(len(mg_coefs)))
    ax.set_xticklabels([f'Firm {e}' for e in mg_coefs.index], rotation=45)
    ax.set_ylabel('Coefficient')
    ax.set_title(f'Coefficient Estimates: {var}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Decision Guide

| Scenario | Preferred Estimator |
|----------|--------------------|
| Slopes homogeneous, entity effects correlated with X | Fixed Effects |
| Slopes homogeneous, entity effects uncorrelated with X | Random Effects |
| Slopes heterogeneous, no long-run structure | Mean Group |
| Long-run homogeneity, short-run heterogeneity | Pooled Mean Group |

**Testing strategy:**
1. Swamy test: Are slopes heterogeneous? If yes, FE/RE are suspect
2. Hausman MG vs PMG: Is long-run homogeneity valid? If yes, PMG is more efficient

---
## 9. Exercises

### Exercise 1: Weighted Mean Group

The MG estimator gives equal weight to each entity. But entities with more precise estimates (lower standard errors) arguably deserve more weight. Fit a weighted MG using the inverse of the entity-specific variance of `value` as weights.

**Hint:** Use the `weights` parameter in `MeanGroupEstimator`.

In [ ]:
# Your code here


### Exercise 2: MG with Simulated Data

Simulate a panel with N=30 entities and T=50 periods where:
- $y_{it} = \alpha_i + \beta_i x_{it} + \varepsilon_{it}$
- $\beta_i \sim N(3, 0.5^2)$ (mild heterogeneity)
- $\alpha_i \sim N(0, 2^2)$

Compare FE and MG estimates. Is the Swamy test significant? Why or why not?

In [ ]:
# Your code here


### Exercise 3: PMG with Different Lags

Fit the PMG model on the Grunfeld data with `lags=1` and `lags=2`. Compare the long-run coefficients. Are they sensitive to the lag specification?

In [ ]:
# Your code here


---

**Solutions** are available in `solutions/01_mean_group_solutions.ipynb`.